# 10 - Operational Prioritization

## Objective

This notebook transforms model predictions into operational recommendations by scoring the September–October validation window, ranking high-risk flights, applying capacity-constrained prioritization logic, and evaluating prioritized selection against a random baseline for RQ4/H4.

Outputs support the Streamlit prioritization tab, the final report, and downstream dashboard preparation.

#### Load project configuration

In [0]:
from config import project_config as cfg

print("Project configuration loaded successfully.")
print(f"Predictions table: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization table: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"Scoring window: {cfg.SCORING_START_DATE} to {cfg.SCORING_END_DATE}")


Project configuration loaded successfully.
Predictions table: workspace.default.flight_predictions
Prioritization table: workspace.default.flight_prioritization_results
Scoring window: 2025-09-01 to 2025-10-31


#### Load the saved model and modeling checkpoints

The selected Spark ML model and the validation-period modeling datasets produced in notebooks 07 and 08 are loaded before batch scoring.

In [0]:
from __future__ import annotations

import json

from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql import functions as F
from utils.model_training import (
    create_feature_hasher_from_manifest,
    hash_modeling_frame,
    load_hist_modeling_table,
)


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run notebooks 07, 08, and 09 before continuing."
        )


def require_file(file_path: str) -> None:
    try:
        dbutils.fs.head(file_path, 1)
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found."
        ) from exc

def require_model_path(model_path: str) -> None:
    try:
        dbutils.fs.ls(model_path)
    except Exception as exc:
        raise RuntimeError(
            f"Required model path '{model_path}' was not found. "
            "Run notebook 08 through the model-save cells."
        ) from exc

require_model_path(cfg.SELECTED_MODEL_PATH)
require_file(cfg.SELECTED_MODEL_METRICS_PATH)
require_file(cfg.MODEL_FEATURE_MANIFEST_PATH)

for table_name in [
    cfg.MODELING_VALIDATION_HIST_TABLE,
    cfg.SHAP_GLOBAL_IMPORTANCE_TABLE,
]:
    require_table(table_name)

feature_manifest = json.loads(
    dbutils.fs.head(cfg.MODEL_FEATURE_MANIFEST_PATH, 1000000)
)
model_metrics = json.loads(
    dbutils.fs.head(cfg.SELECTED_MODEL_METRICS_PATH, 1000000)
)

TARGET_COLUMN = feature_manifest["target_column"]
DECISION_THRESHOLD = float(
    model_metrics.get(
        "selected_validation_threshold",
        cfg.DEFAULT_DECISION_THRESHOLD,
    )
)

final_model = LogisticRegressionModel.load(cfg.SELECTED_MODEL_PATH)
df_validation_hist = load_hist_modeling_table(cfg.MODELING_VALIDATION_HIST_TABLE)
feature_hasher = create_feature_hasher_from_manifest(feature_manifest)
df_validation_hashed = hash_modeling_frame(
    df_validation_hist,
    feature_hasher,
)
global_importance_pdf = (
    spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
    .orderBy(F.col("MeanAbsSHAP").desc())
    .toPandas()
)
top_shap_feature = global_importance_pdf.iloc[0]["Feature"]

print("Saved model and checkpoints loaded successfully.")
print(f"Decision threshold: {DECISION_THRESHOLD:.2f}")
print(f"Model input columns: {len(feature_manifest['model_input_columns'])}")
print(f"Top global SHAP driver: {top_shap_feature}")


[Truncated to first 1 bytes]
[Truncated to first 1 bytes]
Saved model and checkpoints loaded successfully.
Decision threshold: 0.45
Model input columns: 24
Top global SHAP driver: FLIGHT DISTANCE CATEGORY Medium


#### Score the operational review window

The September–October validation window is used as the operational scoring period because it contains completed flights with known delay outcomes and supports RQ4 evaluation without using the final holdout test set.

In [0]:
from pyspark.ml.functions import vector_to_array

scored_predictions = final_model.transform(df_validation_hashed)

scored_predictions = (
    scored_predictions
    .withColumn(
        "delay_probability",
        F.element_at(
            vector_to_array(F.col("probability")),
            2,
        ),
    )
    .withColumn(
        "predicted_delay",
        (
            F.col("delay_probability") >= F.lit(DECISION_THRESHOLD)
        ).cast("double"),
    )
    .select(
        *cfg.MODELING_JOIN_KEY_COLUMNS,
        F.col(TARGET_COLUMN).alias("actual_delay"),
        "predicted_delay",
        "delay_probability",
    )
)

display(scored_predictions.limit(5))
print(f"Scored flights: {scored_predictions.count():,}")
print(f"Operational decision threshold: {DECISION_THRESHOLD:.2f}")

FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,actual_delay,predicted_delay,delay_probability
2025-10-06,AA,1,JFK,LAX,700,1,0.0,0.23619682926483232
2025-10-06,AA,10,LAX,JFK,2204,0,0.0,0.4223314244291485
2025-10-06,AA,1002,MSN,CLT,720,0,0.0,0.21554486633175451
2025-10-06,AA,1003,CLT,MCI,1619,0,1.0,0.4931958609063093
2025-10-06,AA,1003,MCI,CLT,1835,0,1.0,0.5156412536626114


Scored flights: 1,159,898
Operational decision threshold: 0.45


In [0]:
import pandas as pd

from utils.operational_prioritization import (
    add_operational_scores,
    assign_shap_main_drivers,
)


prediction_attributes = df_validation_hist.select(
    *cfg.MODELING_JOIN_KEY_COLUMNS,
    cfg.MONTH_COLUMN,
    cfg.TIME_OF_DAY_COLUMN,
    cfg.SEASON_COLUMN,
)

predictions_frame = (
    scored_predictions.join(
        prediction_attributes,
        on=cfg.MODELING_JOIN_KEY_COLUMNS,
        how="inner",
    )
    .withColumn(
        "flight_label",
        F.concat_ws(
            "-",
            F.col(cfg.AIRLINE_COLUMN),
            F.col(cfg.FLIGHT_NUMBER_COLUMN).cast("string"),
            F.col(cfg.ORIGIN_COLUMN),
            F.col(cfg.DESTINATION_COLUMN),
            F.col(cfg.SCHEDULED_DEPARTURE_COLUMN).cast("string"),
        ),
    )
    .withColumn(
        "scheduled_departure_text",
        F.date_format(
            F.to_timestamp(
                F.lpad(F.col(cfg.SCHEDULED_DEPARTURE_COLUMN).cast("string"), 4, "0"),
                "HHmm",
            ),
            "HH:mm",
        ),
    )
)

scored_count = scored_predictions.count()
joined_count = predictions_frame.count()
if joined_count != scored_count:
    raise ValueError(
        "Prediction join inflated or dropped rows: "
        f"scored={scored_count:,}, joined={joined_count:,}. "
        "Re-run notebooks 06 and 07 after syncing the flight-number join key."
    )

predictions_pdf = predictions_frame.toPandas()
predictions_pdf = predictions_pdf.rename(
    columns={
        cfg.AIRLINE_COLUMN: "airline_code",
        cfg.FLIGHT_NUMBER_COLUMN: "flight_number",
        cfg.ORIGIN_COLUMN: "origin_airport",
        cfg.DESTINATION_COLUMN: "destination_airport",
        cfg.SCHEDULED_DEPARTURE_COLUMN: "scheduled_departure",
        cfg.MONTH_COLUMN: "month_number",
        cfg.TIME_OF_DAY_COLUMN: "departure_window",
        cfg.SEASON_COLUMN: "season",
    }
)
predictions_pdf["shap_main_driver"] = assign_shap_main_drivers(
    predictions_pdf,
    global_importance_pdf,
    fallback_feature=top_shap_feature,
)
predictions_pdf = add_operational_scores(
    predictions_pdf,
    high_threshold=cfg.HIGH_RISK_THRESHOLD,
    critical_threshold=cfg.CRITICAL_RISK_THRESHOLD,
    medium_threshold=cfg.MEDIUM_RISK_THRESHOLD,
)

display(spark.createDataFrame(predictions_pdf).limit(5))
print(f"Operational predictions prepared: {len(predictions_pdf):,}")
print(f"Join check passed: {joined_count:,} scored flights retained.")


FL_DATE,airline_code,flight_number,origin_airport,destination_airport,scheduled_departure,actual_delay,predicted_delay,delay_probability,month_number,departure_window,season,flight_label,scheduled_departure_text,shap_main_driver,risk_level,priority_score,recommendation
2025-09-01,AA,9,ABQ,DFW,1643,1,1.0,0.5516923824654694,9,Afternoon,Fall,AA-9-ABQ-DFW-1643,16:43,SEASON Fall,MEDIUM,55,Increased Operational Monitoring
2025-09-01,AA,315,LAX,MIA,614,0,0.0,0.23898232776902673,9,Morning,Fall,AA-315-LAX-MIA-614,06:14,SEASON Fall,LOW,24,Routine Monitoring
2025-09-01,AA,327,ORD,LGA,600,1,0.0,0.3469746244027818,9,Morning,Fall,AA-327-ORD-LGA-600,06:00,SEASON Fall,MEDIUM,35,Increased Operational Monitoring
2025-09-01,AA,339,DFW,PHX,1840,1,1.0,0.6362259530958443,9,Evening,Fall,AA-339-DFW-PHX-1840,18:40,SEASON Fall,HIGH,64,Priority Operational Review
2025-09-01,AA,345,LAX,MIA,750,0,0.0,0.271852356919845,9,Morning,Fall,AA-345-LAX-MIA-750,07:50,SEASON Fall,LOW,27,Routine Monitoring


Operational predictions prepared: 1,159,898
Join check passed: 1,159,898 scored flights retained.


In [0]:
from utils.operational_prioritization import (
    build_ranking_table,
    compare_prioritization_strategies,
)


prioritization_pool = predictions_pdf[
    predictions_pdf["delay_probability"] >= cfg.PRIORITIZATION_POOL_MIN_PROB
].copy()

ranking_tables = []
evaluation_tables = []

for capacity_k in cfg.CAPACITY_K_OPTIONS:
    ranking = build_ranking_table(
        prioritization_pool,
        capacity_k=capacity_k,
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    ranking["capacity_k"] = capacity_k
    ranking_tables.append(ranking)

    evaluation = compare_prioritization_strategies(
        prioritization_pool,
        capacity_k=capacity_k,
        random_seed=cfg.RANDOM_SEED,
        label_column="actual_delay",
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    evaluation_tables.append(evaluation)

prioritization_results_pdf = pd.concat(ranking_tables, ignore_index=True)
prioritization_evaluation_pdf = pd.concat(evaluation_tables, ignore_index=True)

display(
    spark.createDataFrame(
        prioritization_evaluation_pdf[
            prioritization_evaluation_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K
        ]
    )
)
display(
    spark.createDataFrame(
        prioritization_results_pdf[
            (prioritization_results_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K)
            & (prioritization_results_pdf["selected"])
        ].head(20)
    )
)


capacity_k,strategy,population_size,selected_count,total_delayed_flights,captured_delayed_flights,delay_recall,delay_precision,lift_vs_random
25,Constrained Optimized Selection,79742.0,25.0,28295.0,10.0,3.5341933203746247E-4,0.4,1.127294575013253
25,Top-K Probability Baseline,79742.0,25.0,28295.0,11.0,3.887612652412087E-4,0.44,1.2400240325145784
25,Random Baseline,79742.0,25.0,28295.0,8.0,2.8273546562996996E-4,0.32,0.9018356600106024


FL_DATE,airline_code,flight_number,origin_airport,destination_airport,scheduled_departure,actual_delay,predicted_delay,delay_probability,month_number,departure_window,season,flight_label,scheduled_departure_text,shap_main_driver,risk_level,priority_score,recommendation,selected,priority_rank,capacity_k
2025-10-31,G4,1231,ROA,SFB,2043,1,1.0,0.8551000607583332,10,Evening,Fall,G4-1231-ROA-SFB-2043,20:43,SEASON Fall,CRITICAL,86,Immediate Operational Assessment,true,1,25
2025-09-26,G4,1336,ROA,SFB,2026,1,1.0,0.8450569049304666,9,Evening,Fall,G4-1336-ROA-SFB-2026,20:26,SEASON Fall,CRITICAL,85,Immediate Operational Assessment,true,2,25
2025-09-12,G4,1336,ROA,SFB,2026,1,1.0,0.8450569049304666,9,Evening,Fall,G4-1336-ROA-SFB-2026,20:26,SEASON Fall,CRITICAL,85,Immediate Operational Assessment,true,3,25
2025-09-19,G4,1336,ROA,SFB,2026,0,1.0,0.8450569049304666,9,Evening,Fall,G4-1336-ROA-SFB-2026,20:26,SEASON Fall,CRITICAL,85,Immediate Operational Assessment,true,4,25
2025-10-05,AA,2549,ORD,EWR,2041,0,1.0,0.8243328467674723,10,Evening,Fall,AA-2549-ORD-EWR-2041,20:41,SEASON Fall,CRITICAL,82,Immediate Operational Assessment,true,31,25
2025-10-19,AA,3253,ORD,EWR,2035,0,1.0,0.8224595398617235,10,Evening,Fall,AA-3253-ORD-EWR-2035,20:35,SEASON Fall,CRITICAL,82,Immediate Operational Assessment,true,32,25
2025-10-26,AA,3253,ORD,EWR,2035,1,1.0,0.8224595398617235,10,Evening,Fall,AA-3253-ORD-EWR-2035,20:35,SEASON Fall,CRITICAL,82,Immediate Operational Assessment,true,33,25
2025-10-12,AA,3253,ORD,EWR,2035,1,1.0,0.8224595398617235,10,Evening,Fall,AA-3253-ORD-EWR-2035,20:35,SEASON Fall,CRITICAL,82,Immediate Operational Assessment,true,34,25
2025-10-19,OO,5052,JLN,DEN,1823,0,1.0,0.8198303318959749,10,Evening,Fall,OO-5052-JLN-DEN-1823,18:23,SEASON Fall,CRITICAL,82,Immediate Operational Assessment,true,39,25
2025-10-12,OO,5052,JLN,DEN,1823,0,1.0,0.8198303318959749,10,Evening,Fall,OO-5052-JLN-DEN-1823,18:23,SEASON Fall,CRITICAL,82,Immediate Operational Assessment,true,40,25


#### Validate RQ4 / H4

RQ4 is supported when prioritized selection captures more delayed flights than a random baseline at the same operational capacity K. The comparison below provides the statistical evidence for the final report.

In [0]:
rq4_summary = prioritization_evaluation_pdf.copy()


random_results = (
    rq4_summary[
        rq4_summary["strategy"] == "Random Baseline"
    ][
        [
            "capacity_k",
            "captured_delayed_flights",
        ]
    ]
    .rename(
        columns={
            "captured_delayed_flights":
            "random_captured_delayed_flights"
        }
    )
)

rq4_summary = rq4_summary.merge(
    random_results,
    on="capacity_k",
    how="left",
)


rq4_summary["rq4_supported"] = (
    (
        rq4_summary["strategy"]
        == "Constrained Optimized Selection"
    )
    & (
        rq4_summary["captured_delayed_flights"]
        > rq4_summary["random_captured_delayed_flights"]
    )
)

display(
    spark.createDataFrame(
        rq4_summary
    )
)


default_k_results = rq4_summary[
    rq4_summary["capacity_k"]
    == cfg.DEFAULT_CAPACITY_K
].copy()

optimized_results = default_k_results[
    default_k_results["strategy"]
    == "Constrained Optimized Selection"
]

top_k_results = default_k_results[
    default_k_results["strategy"]
    == "Top-K Probability Baseline"
]

random_results_default = default_k_results[
    default_k_results["strategy"]
    == "Random Baseline"
]


if optimized_results.empty:
    raise ValueError(
        "Constrained Optimized Selection result was not found "
        f"for capacity K={cfg.DEFAULT_CAPACITY_K}."
    )

if top_k_results.empty:
    raise ValueError(
        "Top-K Probability Baseline result was not found "
        f"for capacity K={cfg.DEFAULT_CAPACITY_K}."
    )

if random_results_default.empty:
    raise ValueError(
        "Random Baseline result was not found "
        f"for capacity K={cfg.DEFAULT_CAPACITY_K}."
    )

optimized_row = optimized_results.iloc[0]
top_k_row = top_k_results.iloc[0]
random_row = random_results_default.iloc[0]


print(
    f"RQ4 default-K comparison at "
    f"K={cfg.DEFAULT_CAPACITY_K}:"
)

print(
    "Constrained optimized selection captured "
    f"{int(optimized_row['captured_delayed_flights'])} delayed flights "
    f"with precision "
    f"{optimized_row['delay_precision']:.2%}."
)

print(
    "Top-K probability baseline captured "
    f"{int(top_k_row['captured_delayed_flights'])} delayed flights "
    f"with precision "
    f"{top_k_row['delay_precision']:.2%}."
)

print(
    "Random baseline captured "
    f"{int(random_row['captured_delayed_flights'])} delayed flights "
    f"with precision "
    f"{random_row['delay_precision']:.2%}."
)

rq4_supported_default = (
    optimized_row["captured_delayed_flights"]
    > random_row["captured_delayed_flights"]
)

print(
    f"RQ4 supported at default capacity: "
    f"{rq4_supported_default}"
)

capacity_k,strategy,population_size,selected_count,total_delayed_flights,captured_delayed_flights,delay_recall,delay_precision,lift_vs_random,random_captured_delayed_flights,rq4_supported
10,Constrained Optimized Selection,79742.0,10.0,28295.0,5.0,1.7670966601873123E-4,0.5,1.4091182187665665,3.0,true
10,Top-K Probability Baseline,79742.0,10.0,28295.0,7.0,2.473935324262237E-4,0.7,1.9727655062731932,3.0,false
10,Random Baseline,79742.0,10.0,28295.0,3.0,1.0602579961123873E-4,0.3,0.8454709312599399,3.0,false
25,Constrained Optimized Selection,79742.0,25.0,28295.0,10.0,3.5341933203746247E-4,0.4,1.127294575013253,8.0,true
25,Top-K Probability Baseline,79742.0,25.0,28295.0,11.0,3.887612652412087E-4,0.44,1.2400240325145784,8.0,false
25,Random Baseline,79742.0,25.0,28295.0,8.0,2.8273546562996996E-4,0.32,0.9018356600106024,8.0,false
50,Constrained Optimized Selection,79742.0,50.0,28295.0,26.0,9.188902632974023E-4,0.52,1.465482947517229,24.0,true
50,Top-K Probability Baseline,79742.0,50.0,28295.0,21.0,7.421805972786711E-4,0.42,1.1836593037639158,24.0,false
50,Random Baseline,79742.0,50.0,28295.0,24.0,8.482063968899098E-4,0.48,1.3527534900159037,24.0,false
100,Constrained Optimized Selection,79742.0,56.0,28295.0,29.0,0.001024916062908641,0.5178571428571429,1.4594438694368008,43.0,false


RQ4 default-K comparison at K=25:
Constrained optimized selection captured 10 delayed flights with precision 40.00%.
Top-K probability baseline captured 11 delayed flights with precision 44.00%.
Random baseline captured 8 delayed flights with precision 32.00%.
RQ4 supported at default capacity: True


At higher capacity levels, the constrained optimization strategy may select fewer than \(K\) flights when airline and origin-airport diversification limits prevent additional feasible selections. For example, at \(K=100\), the optimizer selected 56 flights. Therefore, comparisons at this capacity should be interpreted with consideration of the binding diversification constraints.

#### Save operational outputs

In [0]:
predictions_df = spark.createDataFrame(predictions_pdf)
prioritization_results_df = spark.createDataFrame(prioritization_results_pdf)
prioritization_evaluation_df = spark.createDataFrame(prioritization_evaluation_pdf)

(
    predictions_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.PREDICTIONS_DELTA_PATH)
)
(
    predictions_df.writeTo(cfg.PREDICTIONS_TABLE).using("delta").createOrReplace()
)

(
    prioritization_results_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.PRIORITIZATION_RESULTS_PATH)
)
(
    prioritization_results_df.writeTo(cfg.PRIORITIZATION_RESULTS_TABLE)
    .using("delta")
    .createOrReplace()
)

(
    prioritization_evaluation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.PRIORITIZATION_EVALUATION_PATH)
)
(
    prioritization_evaluation_df.writeTo(cfg.PRIORITIZATION_EVALUATION_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Operational prioritization outputs saved successfully.")
print(f"Predictions table: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization results: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"Prioritization evaluation: {cfg.PRIORITIZATION_EVALUATION_TABLE}")


Operational prioritization outputs saved successfully.
Predictions table: workspace.default.flight_predictions
Prioritization results: workspace.default.flight_prioritization_results
Prioritization evaluation: workspace.default.flight_prioritization_evaluation
